# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya  Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the [FAIR^2 dataset](https://sen.science/doi/10.71728/senscience.y7m0-f273) using the `mlcroissant` library, following the Croissant schema.

### Dataset Source
The dataset source is defined as a Croissant schema at the following URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Install mlcroissant in the current environment
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Display dataset title and description
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets and fields, referencing each by their `@id`.

We'll enumerate all record sets and their fields.

In [ ]:
# List record sets and fields by @id
print("Available record sets and their fields (referenced by '@id'):")
record_sets = []
for rs in metadata.record_sets:
    print(f"\nRecord set: {rs.id}")
    record_sets.append(rs.id)
    for field in rs.fields:
        print(f"  - Field: {field.id} (type: {getattr(field, 'data_type', 'Unknown')})")

Let's inspect a few records from each record set to see their structure and data.
We use the `@id` of the record set when loading records.

In [ ]:
for record_set_id in record_sets:
    print(f"\nSample records from record set: {record_set_id}")
    for idx, rec in enumerate(dataset.records(record_set=record_set_id)):
        print(rec)
        if idx >= 2:
            break  # Print first 3 rows per record set

## 3. Data Extraction
Let's load the records from each record set into pandas DataFrames for analysis, referencing each record set by its `@id`.

We will then preview the available columns (fields) in the first record set.

In [ ]:
# Convert all record sets into DataFrames, referenced by @id
dataframes = dict()

for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

main_record_set_id = record_sets[0] if record_sets else None
if main_record_set_id and not dataframes[main_record_set_id].empty:
    print(f"Columns in record set {main_record_set_id}:\n{list(dataframes[main_record_set_id].columns)}\n")
    display(dataframes[main_record_set_id].head())
else:
    print("No record sets or data loaded.")

## 4. Exploratory Data Analysis (EDA)
Let's process a numeric field. We'll select a numeric field for analysis—referenced by its `@id`—and demonstrate filtering, normalization, and grouping.

In [ ]:
# Identify a numeric field from the first DataFrame (if any numeric fields found)
import numpy as np

numeric_fields = []
df = dataframes[main_record_set_id] if main_record_set_id else pd.DataFrame()
for col in df.columns:
    # Try to infer if a column is numeric by checking its dtype or trying a conversion
    sample = df[col].dropna().head(10)
    if not sample.empty:
        try:
            pd.to_numeric(sample, errors='raise')
            numeric_fields.append(col)
        except Exception:
            pass

if not numeric_fields:
    print("No numeric fields available for EDA.")
else:
    numeric_field_id = numeric_fields[0]
    print(f"Selected numeric field for analysis: {numeric_field_id}")

    # Convert the column to numeric (if not already)
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

    # Example threshold: 10
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records where {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    )
    print(f"Normalized {numeric_field_id}:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # If there is a categorical field, group by it
    group_field_id = None
    for col in df.columns:
        if col != numeric_field_id and df[col].nunique() > 1 and df[col].nunique() < len(df)/2:
            group_field_id = col
            break
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
        display(grouped_df.head())

## 5. Visualization
Visualize the distribution of the selected numeric field (using its `@id`) and its relationship with a categorical variable (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not numeric_fields:
    print("No numeric fields to visualize.")
else:
    # Histogram of the selected numeric field
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of '{numeric_field_id}'")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # Boxplot of numeric field by group if group field exists
    if 'group_field_id' in locals() and group_field_id:
        plt.figure(figsize=(10, 4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
We successfully loaded the dataset using the Croissant schema and explored its structure using `mlcroissant`. The notebook demonstrated how to access record sets, examine fields by their `@id`, filter and normalize numeric fields, and visualize results—all in a reproducible FAIR-compliant workflow.

Explore other record sets, fields, and derived analyses by updating record set and field `@id` values as needed in this template!